<a href="https://colab.research.google.com/github/Dhanapal-Angamuthu/DeepLearning/blob/main/NLP/TFIDF/NLP_Spam_Classification_using_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv('/content/HamSpamMsgCollections.txt', sep='\t', names=['label', 'message'])
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
corpus = []
for i in range(0, len(df)):
  words = re.sub('[^a-zA-Z]', ' ', df['message'][i])
  words = words.lower()
  words = words.split()

  words = [ps.stem(word) for word in words if not word in stopwords.words('english')]
  words = ' '.join(words)
  corpus.append(words)
  corpus

In [26]:
## Creating TF-IDF model for spam classification
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X = tv.fit_transform(corpus).toarray()

In [21]:
'''
 This line uses the get_dummies function from the pandas library to perform one-hot encoding on the 'label' column of the DataFrame df. One-hot encoding converts categorical variables (like 'ham' and 'spam') into a numerical representation that can be used by machine learning algorithms. It creates new columns for each unique value in the 'label' column and assigns a 1 or 0 to indicate the presence of that value.
y = y.iloc[:, 1].values: After one-hot encoding, the resulting y will be a DataFrame with two columns ('ham' and 'spam'). This line selects the second column (index 1), which corresponds to 'spam', using .iloc[:, 1]. .values then extracts the values from this column as a NumPy array. This effectively creates a binary target variable where 1 represents 'spam' and 0 represents 'ham'.
'''
y = pd.get_dummies(df['label'])
y = y.iloc[:, 1].values

In [27]:
## Training and Testing set split
from sklearn.model_selection import train_test_split
X_test, X_train, y_test, y_train = train_test_split(X, y, test_size=0.2, random_state=0)

In [28]:
'''
 Multinomial Naive Bayes is a classification algorithm that is often used for text classification tasks, like spam detection.
spam_detect_model = MultinomialNB().fit(X_train, y_train):
This line creates an instance of the MultinomialNB classifier and then trains it using the training data.
MultinomialNB(): This creates an instance of the classifier with default parameters.
.fit(X_train, y_train): This method trains the model. X_train contains the features (the TF-IDF vectors of the messages) and y_train contains the corresponding labels (whether the message is spam or not) for the training set. The fit method learns the relationships between the features and the labels from the training data. The trained model is then assigned to the variable spam_detect_model.
'''
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [32]:
## Prediction
y_predict = spam_detect_model.predict(X_test)

In [33]:
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test, y_predict)
score

0.9549024007179717

In [34]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(y_test, y_predict))
confusion_m = confusion_matrix(y_test, y_predict)
confusion_m

              precision    recall  f1-score   support

       False       0.95      1.00      0.97      3870
        True       1.00      0.66      0.79       587

    accuracy                           0.95      4457
   macro avg       0.98      0.83      0.88      4457
weighted avg       0.96      0.95      0.95      4457



array([[3870,    0],
       [ 201,  386]])

In [35]:
## Another option is to use RandomForest Classifier model
from sklearn.ensemble import RandomForestClassifier
spam_detect_model = RandomForestClassifier().fit(X_train, y_train)

In [36]:
y_pred = spam_detect_model.predict(X_test)

In [38]:
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.9670181736594121
[[3861    9]
 [ 138  449]]
              precision    recall  f1-score   support

       False       0.97      1.00      0.98      3870
        True       0.98      0.76      0.86       587

    accuracy                           0.97      4457
   macro avg       0.97      0.88      0.92      4457
weighted avg       0.97      0.97      0.97      4457

